# Base final para las redes bayesianas — ENMT

**Proyecto:** transporte_UNAM · **Etapa:** 03 — construccion de la base analitica final
**Entrada:** `data/processed/enmt_limpio.csv` + `data/processed/diccionario.json` (salidas de `01_limpieza_enmt.ipynb`)
**Salidas:** `enmt_bn_final.csv`, `enmt_bn_codebook.csv` y `enmt_bn_q1..q4.csv`

---

## Que hace este notebook

Cierra los puntos abiertos que dejo `02_seleccion_columnas.ipynb`, **deriva los nodos que no existen
como columna** en la encuesta, los recodifica a las categorias del modelo y exporta una base
autocontenida, lista para leerse en R.

Aqui **no se construyen DAGs ni se hace inferencia**: eso ocurre despues en R, a partir de
`enmt_bn_final.csv`. El objetivo de este notebook es que ese paso empiece con los datos ya resueltos.

### Los 4 queries que la base debe poder responder

| | Query | Nodos |
|---|---|---|
| Q1 | Usuarios de metro vs. resto de TP: percepcion de seguridad y efectividad | `grupo_tp` -> `perc_seguridad`, `perc_eficiencia` |
| Q2 | Patrones/autoempleados vs. profesionistas: percepcion del costo | `ocupacion` -> `perc_costo` |
| Q3 | Escolaridad -> modo principal | `escolaridad` -> `modo_principal` |
| Q4 | Entre usuarios de TP: asalto en TP segun ingreso | `ingreso_gpo` -> `asalto_tp` |

### Los tres nodos que hubo que derivar

Ninguno de estos existe como columna en la encuesta:

1. **`ocupacion`** — la ocupacion vive en el *roster* del hogar (`h21_*`, una columna por integrante),
   no a nivel del informante. Hay que averiguar **que renglon del roster es la persona entrevistada**.
2. **`modo_principal`** — se deduce de la bateria `p1a_*` (frecuencia de uso de 22 modos).
3. **`grupo_tp` / `usa_tp` / `usa_metro`** — los universos de Q1 y Q4.

## 1. Configuracion y carga

In [1]:
import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

# Raiz robusta: mismo criterio que 01_limpieza y 02_seleccion
BASE = Path.cwd()
if not (BASE / "data").is_dir():
    BASE = BASE.parent
DIR_PROC = BASE / "data" / "processed"
RUTA_LIMPIO = DIR_PROC / "enmt_limpio.csv"
RUTA_DICC = DIR_PROC / "diccionario.json"

df = pd.read_csv(RUTA_LIMPIO, low_memory=False)
DICC = json.loads(RUTA_DICC.read_text(encoding="utf-8"))["variables"]


def etiqueta(col):
    # Etiqueta de la variable segun el codebook.
    return DICC.get(col, {}).get("etiqueta", "")


def valores(col):
    # Catalogo {codigo: etiqueta} de la variable, con las claves como int.
    return {int(k): v for k, v in DICC.get(col, {}).get("valores", {}).items()
            if k.lstrip("-").isdigit()}


def familia(prefijo):
    # Columnas tipo `p1a_3` de una familia, ordenadas por su indice numerico.
    cols = [c for c in df.columns if re.fullmatch(rf"{prefijo}_\d+", c)]
    return sorted(cols, key=lambda c: int(c.split("_")[-1]))


COLS_P1A = familia("p1a")                                     # los 22 modos de transporte
RENG_ROSTER = [int(c.split("_")[1]) for c in familia("h10")]  # renglones con sexo y edad
RENG_OCUP = [int(c.split("_")[1]) for c in familia("h21")]    # renglones con ocupacion

print(f"Base completa : {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
print(f"Diccionario   : {len(DICC):,} variables")
print(f"Modos p1a_*   : {len(COLS_P1A)}")
print(f"Roster        : sexo/edad en los renglones {RENG_ROSTER}")
print(f"                ocupacion solo en {RENG_OCUP} (la limpieza tiro los demas por vacios)")

Base completa : 1,191 filas x 556 columnas
Diccionario   : 556 variables
Modos p1a_*   : 22
Roster        : sexo/edad en los renglones [1, 2, 3, 4, 5, 6, 7, 8]
                ocupacion solo en [1, 2, 3, 4, 5, 6] (la limpieza tiro los demas por vacios)


## 2. Lo que dice el codebook

Esta seccion **resuelve los cuatro puntos abiertos** que dejo el notebook 02. No transforma nada:
solo imprime lo que el diccionario dice de cada variable que va a ser nodo, para que las decisiones
de las secciones siguientes queden justificadas y sean auditables.

In [2]:
def ficha(col):
    v = DICC.get(col)
    if v is None:
        print(f"### {col} — NO ESTA en enmt_limpio.csv\n")
        return
    print(f"### {col} — {v['etiqueta'][:100]}")
    print(f"    nulos: {v['pct_nulos']}%   valores: {v['valores']}\n")


for c in ["escol", "ing_ind", "p17_1", "p17_3", "p17_4", "p25_1_2", "h21_1"]:
    ficha(c)

### escol — Escolaridad
    nulos: 0.08%   valores: {'2': 'Primaria', '3': 'Secundaria', '4': 'Preparatoria o Bachillerato', '5': 'Universidad o Posgrado', '8': 'NS', '9': 'NC'}

### ing_ind — Ingreso Individual
    nulos: 0.08%   valores: {'2': 'Menos de $2,047.49 (Menos de 1 SM)', '3': 'De $2,047.50 a $4,095.00 (Más de 1 hasta 2 SM)', '4': 'De $4,095.01 a $ 6,142.50 (Más de 2 hasta 3 SM)', '5': 'Más de $ 6,142.50 (Más de 3 SM)', '999998': 'NS', '999999': 'NS/NC'}

### p17_1 — 17 Usted, ¿cómo considera el transporte público en su localidad  o ciudad? Eficiente  ó Ineficiente
    nulos: 0.0%   valores: {'1': 'Eficiente', '2': 'Ineficiente'}

### p17_3 — 17 Usted, ¿cómo considera el transporte público en su localidad  o ciudad? Barato  ó Caro
    nulos: 0.0%   valores: {'1': 'Barato', '2': 'Caro'}

### p17_4 — 17 Usted, ¿cómo considera el transporte público en su localidad  o ciudad? Seguro  ó Inseguro
    nulos: 0.0%   valores: {'1': 'Seguro', '2': 'Inseguro'}

### p25_1_2 — 25.1 ¿Uste

### Lo que confirma el codebook

- **`p25_1_2` es exactamente el nodo de Q4**: *"¿Usted ha sido o no ha sido victima de alguno de los
  siguientes delitos en sus viajes cotidianos? **Asalto en transporte publico**"*.
- **`p17_*` es la mejor fuente de percepcion**: las cinco son binarias (`1` = polo positivo) y tienen
  **cero faltantes**. Se descarta la bateria alterna `p1c_*`, que llega a 97 % de "no aplica" en
  varios modos y cuyo sufijo de costo solo existe para 11 de los 22 modos.
- **`h21` resuelve Q2 por si sola.** Contiene `1 = Profesionista`, `12 = Trabajador por cuenta propia`
  y `13 = Patron`. Al ser una unica variable, sus categorias son mutuamente excluyentes: **no hay
  traslape que desempatar** entre "posicion en el trabajo" y "ocupacion", como se temia al planear.
  (`h22` resulto ser la *frecuencia de pago* — semanal, quincenal, mensual — y no se usa.)
- **`escol` es un nivel codificado**, no anios de estudio.

### Codigos sin documentar

El codebook omite la primera fila del catalogo en varias variables — el mismo tipo de hueco que
motivo la regla B2 del notebook 01. Afecta a cuatro variables que aqui son nodo:

In [3]:
SIN_DOC = {}
for c in ["region", "tam_loc", "escol", "ing_ind", "edad_1"]:
    obs = {int(x) for x in df[c].dropna().unique()}
    huerfanos = sorted(obs - set(valores(c)))
    SIN_DOC[c] = huerfanos
    n = int(df[c].isin(huerfanos).sum()) if huerfanos else 0
    print(f"  {c:9} codigos sin etiqueta: {huerfanos if huerfanos else '(ninguno)'}"
          f"{f'  -> {n:,} casos' if huerfanos else ''}")

print("\n--- Evidencia para escol=1: cruzarlo contra sd4 (nivel de instruccion detallado) ---")
tab = pd.crosstab(df["escol"], df["sd4"])
tab.columns = [f"{int(k)}. {valores('sd4').get(int(k), '?')[:22]}" for k in tab.columns]
display(tab)

  region    codigos sin etiqueta: [1]  -> 348 casos
  tam_loc   codigos sin etiqueta: [1]  -> 622 casos
  escol     codigos sin etiqueta: [1]  -> 146 casos
  ing_ind   codigos sin etiqueta: [0]  -> 589 casos
  edad_1    codigos sin etiqueta: (ninguno)

--- Evidencia para escol=1: cruzarlo contra sd4 (nivel de instruccion detallado) ---


,1. Ninguno,2. Preescolar,3. Primaria,4. Secundaria,5. Carrera tecnica con se,6. Preparatoria o bachill,7. Carrera tecnica con pr,8. Normal,9. Profesional,10. Maestria o doctorado,11. Carrera secretarial co
escol,,,,,,,,,,,
1.0,44,9,88,0,0,0,0,0,0,0,1
2.0,0,0,133,42,4,0,0,0,0,0,1
3.0,0,0,0,312,77,51,14,0,0,0,0
4.0,0,0,0,0,0,184,80,11,32,3,0
5.0,0,0,0,0,0,0,0,7,75,6,0


Los cuatro huecos se resuelven asi, y cada decision queda anotada en el codebook de salida:

| Variable | Codigo | Lectura | En que se apoya |
|---|---|---|---|
| `escol` | `1` | Sin primaria completa | El cruce de arriba: `escol=1` agrupa a quienes en `sd4` declararon *Ninguno*, *Preescolar* o *Primaria* sin haberla concluido, mientras que `escol=2` ("Primaria") son los que si la terminaron. |
| `ing_ind` | `0` | Sin ingreso declarado | Lo documenta el README de la etapa de limpieza. **No es un faltante**: es una respuesta con significado, y por eso entra como la categoria mas baja en lugar de volverse `NA`. |
| `tam_loc` | `1` | 100 000 y mas habitantes | Los codigos `2`, `3` y `4` son tramos descendentes (15 000-99 999, 2 500-14 999, 1-2 499); el `1` solo puede ser el tramo superior. |
| `region` | `1` | *sin resolver* | Las etiquetas conocidas (Metropolitana, Norte, Sur) no permiten deducirlo. Se conserva como `region_1_sin_documentar` en vez de inventarle un nombre. `region` es covariable de contexto, no nodo de ningun query. |

## 3. Enlace del informante con el roster del hogar

El obstaculo que bloqueaba Q2. La ocupacion (`h21_*`) esta en el *roster*: una columna por
integrante del hogar. Para saber cual de esas columnas corresponde a la persona entrevistada hay que
identificar **su renglon**, y la encuesta no trae esa variable.

`h8_*`, que traia los nombres de pila, se elimino en la limpieza por ser dato personal — asi que ese
camino esta cerrado, y ademas nunca hubiera servido para identificar al informante.

**La solucion:** el roster si trae **sexo (`h10_*`) y edad (`h11_*`)** de cada integrante, y del
informante conocemos ambos (`sexo`, `sd2`). Buscando el renglon cuyo par (sexo, edad) coincide se
identifica a la persona. Solo se acepta cuando la coincidencia es **unica**: si dos integrantes
tienen el mismo sexo y la misma edad, el caso se marca como ambiguo y se deja en `NA` en lugar de
elegir uno al azar.

In [4]:
coincide = pd.DataFrame(
    {r: (df[f"h10_{r}"] == df["sexo"]) & (df[f"h11_{r}"] == df["sd2"]) for r in RENG_ROSTER}
)
n_coincidencias = coincide.sum(axis=1)

df["renglon_informante"] = (
    coincide.idxmax(axis=1).where(n_coincidencias == 1).astype("Int64")
)
df["match_informante"] = np.select(
    [n_coincidencias == 1, n_coincidencias > 1],
    ["unico", "ambiguo"],
    default="sin_match",
)

print(df["match_informante"].value_counts().to_frame("casos").to_string())
print(f"\nTasa de identificacion: {(df['match_informante'] == 'unico').mean() * 100:.1f}%")
print("\nRenglon del informante:")
print(df["renglon_informante"].value_counts(dropna=False).sort_index().to_frame("casos").to_string())

sin_ocup = int(df["renglon_informante"].isin([r for r in RENG_ROSTER if r not in RENG_OCUP]).sum())
print(f"\nIdentificados en un renglon sin columna de ocupacion: {sin_ocup} caso(s)")
print("  (h21_* sobrevive solo para los primeros renglones; estos quedan sin ocupacion)")

                  casos
match_informante       
unico              1185
sin_match             4
ambiguo               2

Tasa de identificacion: 99.5%

Renglon del informante:
                    casos
renglon_informante       
1                     595
2                     394
3                     149
4                      38
5                       7
6                       1
7                       1
<NA>                    6

Identificados en un renglon sin columna de ocupacion: 1 caso(s)
  (h21_* sobrevive solo para los primeros renglones; estos quedan sin ocupacion)


## 4. Nodo `ocupacion` (Q2)

Con el renglon identificado, se lee `h21_{renglon}` fila por fila.

**Por que se agrupan patrones con autoempleados.** El query contrasta *patrones* contra
*profesionistas*, pero los patrones (`h21 = 13`) son **muy pocos**: no alcanzan para estimar una
distribucion condicional estable. Se agrupan con los trabajadores por cuenta propia (`h21 = 12`),
que comparten la caracteristica relevante para la pregunta — **no reciben un salario de un tercero**
— y juntos forman una categoria de tamanio utilizable. La celda de abajo imprime los conteos
exactos para que el tamanio de cada grupo quede a la vista.

El codigo `97` ("Ocupaciones insuficientemente especificadas") se manda a `NA`: no dice cual es la
ocupacion, asi que no puede alimentar un nodo de ocupacion. Los `NA` de quienes no trabajan se
conservan como tales — su ausencia de ocupacion es real, no un dato perdido.

In [5]:
def del_renglon(prefijo, renglones):
    # Toma `prefijo_r` de cada fila, donde r es el renglon del informante.
    out = pd.Series(np.nan, index=df.index, dtype="float64")
    for r in renglones:
        marca = df["renglon_informante"] == r
        out[marca] = df.loc[marca, f"{prefijo}_{r}"]
    return out


df["h21_informante"] = del_renglon("h21", RENG_OCUP)

CAT_H21 = valores("h21_1")
tabla = df["h21_informante"].value_counts().sort_index().to_frame("casos")
tabla["etiqueta"] = [CAT_H21.get(int(k), "(sin etiqueta)") for k in tabla.index]
print(f"Ocupacion del informante — {int(df['h21_informante'].notna().sum()):,} valores no nulos\n")
display(tabla)

PROFESIONISTA = [1]        # Profesionista
PATRON_AUTOEMP = [12, 13]  # Trabajador por cuenta propia, Patron
NO_ESPECIFICADA = [97]     # Ocupaciones insuficientemente especificadas -> NA

df["ocupacion"] = np.select(
    [
        df["h21_informante"].isin(PROFESIONISTA),
        df["h21_informante"].isin(PATRON_AUTOEMP),
        df["h21_informante"].isin(NO_ESPECIFICADA) | df["h21_informante"].isna(),
    ],
    ["profesionista", "patron_autoempleado", None],
    default="otro",
)
df["ocupacion"] = df["ocupacion"].replace({"None": None, None: np.nan})

print("\nNodo `ocupacion`:")
display(df["ocupacion"].value_counts(dropna=False).to_frame("casos"))
print(f"Universo de Q2 (patron_autoempleado + profesionista): "
      f"{int(df['ocupacion'].isin(['patron_autoempleado', 'profesionista']).sum()):,} casos")

Ocupacion del informante — 699 valores no nulos



,casos,etiqueta
h21_informante,,
1.0,62,Profesionista
2.0,55,Tecnico
3.0,23,Trabajador de la educacion
4.0,39,"Trabajador en actividades agricolas, ganadera..."
5.0,69,Trabajador en actividades de reparacion y man...
6.0,58,Trabajador en actividades administrativas
7.0,121,Comerciante
8.0,45,Empleado de comercio y agente de ventas
9.0,24,Vendedor ambulante y trabajador ambulante en ...



Nodo `ocupacion`:


,casos
ocupacion,
otro,552
NaN,502
patron_autoempleado,75
profesionista,62


Universo de Q2 (patron_autoempleado + profesionista): 137 casos


## 5. Modo principal y universos de transporte publico

`p1a_*` recoge, para 22 modos, si la persona lo usa `1` cotidianamente, `2` ocasionalmente o
`3` nunca. Los modos se agrupan leyendo **las etiquetas del diccionario**, no por el numero de
columna, para que el mapeo siga siendo correcto si el orden cambia.

In [6]:
def sin_acentos(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()


def grupo_modo(lbl):
    # Agrupa un modo de transporte a partir de su etiqueta en el codebook.
    l = sin_acentos(lbl)
    if "tren urbano" in l:                                     return "metro"
    if re.search(r"\bbrt\b|metrobus|electrico|tranvia|trolebus", l): return "tp_masivo"
    if re.search(r"\btren\b", l):                              return "tp_masivo"
    if "foraneo" in l:                                         return "foraneo"
    if re.search(r"camion|microbus|colectivo|combi", l):       return "tp_concesionado"
    if "taxi" in l:                                            return "taxi"
    if re.search(r"automovil|motocicleta|cuatrimoto", l):      return "auto_moto"
    if re.search(r"bicicleta|triciclo|patines|patineta", l):   return "activo"
    return "otro"


def solo_modo(lbl):
    # Quita el tronco de la pregunta y deja el nombre del modo.
    return re.sub(r"^.*?\?\s*", "", lbl).strip()


MODO_GRUPO = {c: grupo_modo(etiqueta(c)) for c in COLS_P1A}

resumen_modos = pd.DataFrame({
    "modo": [solo_modo(etiqueta(c)) for c in COLS_P1A],
    "grupo": [MODO_GRUPO[c] for c in COLS_P1A],
    "cotidiano": [int((df[c] == 1).sum()) for c in COLS_P1A],
    "ocasional": [int((df[c] == 2).sum()) for c in COLS_P1A],
}, index=COLS_P1A)
display(resumen_modos)

print("\nMapeo de modos verificado.")

,modo,grupo,cotidiano,ocasional
p1a_1,Tren,tp_masivo,6,29
p1a_2,"Tren urbano (metro, suburbano, tren ligero)",metro,63,147
p1a_3,"Transporte eléctrico (tranvía, trolebús)",tp_masivo,8,49
p1a_4,Camión/microbús,tp_concesionado,453,400
p1a_5,"Colectivo (combi, camioneta, minivan)",tp_concesionado,319,285
p1a_6,Autobús foráneo,foraneo,58,379
p1a_7,"BRT (Metrobús, Optibús, Macrobús, RUTA)",tp_masivo,25,61
p1a_8,Taxi,taxi,140,529
p1a_9,Bicitaxi / mototaxi,taxi,59,86
p1a_10,Transporte escolar / de personal,otro,18,17



Mapeo de modos verificado.


### El desempate del modo principal

Una persona puede declarar **varios modos con la misma frecuencia maxima** (usar a diario el camion
y el automovil, por ejemplo), y ahi hay que elegir uno.

El notebook 02 resolvia esto con `idxmin` sobre las columnas ordenadas alfabeticamente, lo que da un
orden `p1a_1, p1a_10, p1a_11, p1a_12, p1a_2, ...`: el ganador terminaba dependiendo del **nombre de
la columna** y no del dato. En la practica `p1a_12` (automovil) le ganaba siempre a `p1a_4`
(camion), aunque el camion sea mucho mas usado.

Aqui el desempate usa una prioridad **derivada de los propios datos**: entre modos empatados gana el
grupo con mas usuarios cotidianos en la muestra. Es reproducible y no depende de un criterio elegido
a mano. Los casos empatados quedan marcados en `modo_principal_empate` por si conviene excluirlos.

In [7]:
uso_cotidiano = {}
for c in COLS_P1A:
    uso_cotidiano[MODO_GRUPO[c]] = uso_cotidiano.get(MODO_GRUPO[c], 0) + int((df[c] == 1).sum())

PRIORIDAD_MODOS = sorted(uso_cotidiano, key=lambda g: -uso_cotidiano[g])
RANGO = {g: i for i, g in enumerate(PRIORIDAD_MODOS)}
print("Prioridad de desempate (por usuarios cotidianos en la muestra):")
for g in PRIORIDAD_MODOS:
    print(f"  {RANGO[g] + 1}. {g:17} {uso_cotidiano[g]:,} usuarios cotidianos")

frec = df[COLS_P1A]
minimo = frec.min(axis=1)


def resolver(fila):
    m = fila.min()
    if pd.isna(m) or m >= 3:      # nadie usa ningun modo ni cotidiana ni ocasionalmente
        return pd.NA, False
    cands = {MODO_GRUPO[c] for c in COLS_P1A if fila[c] == m}
    return min(cands, key=lambda g: RANGO[g]), len(cands) > 1


resuelto = frec.apply(resolver, axis=1)
df["modo_principal"] = [r[0] for r in resuelto]
df["modo_principal_empate"] = [r[1] for r in resuelto]

print(f"\nFilas con empate resuelto por prioridad: {int(df['modo_principal_empate'].sum()):,} "
      f"de {int(df['modo_principal'].notna().sum()):,} con modo principal")
display(df["modo_principal"].value_counts(dropna=False).to_frame("casos"))

Prioridad de desempate (por usuarios cotidianos en la muestra):
  1. tp_concesionado   772 usuarios cotidianos
  2. auto_moto         280 usuarios cotidianos
  3. taxi              199 usuarios cotidianos
  4. activo            88 usuarios cotidianos
  5. otro              64 usuarios cotidianos
  6. metro             63 usuarios cotidianos
  7. foraneo           58 usuarios cotidianos
  8. tp_masivo         39 usuarios cotidianos



Filas con empate resuelto por prioridad: 454 de 1,185 con modo principal


,casos
modo_principal,
tp_concesionado,819
auto_moto,214
taxi,85
activo,25
otro,19
metro,10
foraneo,8
<NA>,6
tp_masivo,5


### Universos de TP: uso, no modo principal

Definir "usuario de transporte publico" por el **modo principal** deja fuera a mucha gente: quien usa
el metro a diario pero maneja mas quedaria clasificado como no-usuario, aunque tenga una opinion
formada del TP. Como Q1 y Q4 preguntan por la experiencia con el transporte publico, el universo se
define por **uso** (cotidiano u ocasional), que es lo que la pregunta necesita.

El **autobus foraneo** queda fuera del TP urbano: es transporte interurbano, y `p17_*` pregunta
explicitamente por *"el transporte publico en su localidad o ciudad"*. Los taxis tambien quedan
fuera por no ser transporte publico de linea.

In [8]:
TP_URBANO = {"metro", "tp_masivo", "tp_concesionado"}
cols_metro = [c for c in COLS_P1A if MODO_GRUPO[c] == "metro"]
cols_tp = [c for c in COLS_P1A if MODO_GRUPO[c] in TP_URBANO]

df["usa_metro"] = df[cols_metro].isin([1, 2]).any(axis=1)
df["usa_tp"] = df[cols_tp].isin([1, 2]).any(axis=1)
df["grupo_tp"] = np.where(df["usa_metro"], "metro",
                          np.where(df["usa_tp"], "otro_tp", "no_tp"))

print(f"Modos que cuentan como TP urbano: {[solo_modo(etiqueta(c)) for c in cols_tp]}\n")
display(df["grupo_tp"].value_counts().to_frame("casos"))
print(f"\nUniverso de TP (Q1 y Q4): {int(df['usa_tp'].sum()):,} personas")
print("Nota: 'metro' incluye a quien usa metro y ademas otro TP; son los usuarios de metro.")

Modos que cuentan como TP urbano: ['Tren', 'Tren urbano (metro, suburbano, tren ligero)', 'Transporte eléctrico (tranvía, trolebús)', 'Camión/microbús', 'Colectivo (combi, camioneta, minivan)', 'BRT (Metrobús, Optibús, Macrobús, RUTA)']



,casos
grupo_tp,
otro_tp,807
metro,210
no_tp,174



Universo de TP (Q1 y Q4): 1,017 personas
Nota: 'metro' incluye a quien usa metro y ademas otro TP; son los usuarios de metro.


## 6. Recodificacion de los nodos

Cada nodo pasa de codigo numerico a **categoria de texto**. Los mapas van escritos a la vista en el
codigo, no ocultos en un archivo aparte, para que cualquiera pueda discutir un corte concreto.

Las categorias se exportan como texto porque el destino es R: `read.csv(stringsAsFactors = TRUE)`
las convierte en factores directamente, sin tener que traducir codigos del otro lado.

In [9]:
NODOS = {}   # nombre -> (serie, orden de niveles, origen, queries)


def registrar(nombre, serie, niveles, origen, queries):
    NODOS[nombre] = (serie, niveles, origen, queries)
    df[nombre] = serie


# --- escolaridad (Q3) --------------------------------------------------------
# 1 = sin primaria completa (sin etiqueta en el codebook, deducido con sd4)
registrar(
    "escolaridad",
    df["escol"].map({1: "basica", 2: "basica", 3: "basica",
                     4: "media_superior", 5: "superior"}),
    ["basica", "media_superior", "superior"],
    "escol {1,2,3} / {4} / {5}",
    "Q3",
)

# --- ingreso (Q4) ------------------------------------------------------------
# ing_ind es un codigo ordinal de rango, NO un monto: promediarlo no significa nada.
# El 0 ("sin ingreso declarado") es respuesta real y entra como la categoria mas baja.
registrar(
    "ingreso_gpo",
    df["ing_ind"].map({0: "sin_ingreso", 2: "hasta_2_sm", 3: "hasta_2_sm",
                       4: "mas_de_2_sm", 5: "mas_de_2_sm"}),
    ["sin_ingreso", "hasta_2_sm", "mas_de_2_sm"],
    "ing_ind {0} / {2,3} / {4,5}",
    "Q4",
)
registrar(
    "ingreso_bin",
    df["ing_ind"].map({0: "bajo", 2: "bajo", 3: "bajo", 4: "alto", 5: "alto"}),
    ["bajo", "alto"],
    "ing_ind {0,2,3} / {4,5}",
    "Q4 (respaldo)",
)

# --- percepcion del TP (Q1, Q2) ---------------------------------------------
for nodo, col, niveles, q in [
    ("perc_seguridad", "p17_4", ["seguro", "inseguro"], "Q1"),
    ("perc_eficiencia", "p17_1", ["eficiente", "ineficiente"], "Q1"),
    ("perc_costo", "p17_3", ["barato", "caro"], "Q2"),
]:
    registrar(nodo, df[col].map({1: niveles[0], 2: niveles[1]}), niveles, col, q)

# --- victimizacion (Q4) ------------------------------------------------------
registrar("asalto_tp", df["p25_1_2"].map({1: "si", 2: "no"}), ["no", "si"], "p25_1_2", "Q4")

# --- derivados ---------------------------------------------------------------
registrar("ocupacion", df["ocupacion"],
          ["patron_autoempleado", "profesionista", "otro"], "h21 del informante", "Q2")
registrar("grupo_tp", df["grupo_tp"], ["metro", "otro_tp", "no_tp"], "p1a_* (uso)", "Q1")
registrar("modo_principal", df["modo_principal"], PRIORIDAD_MODOS, "p1a_* (frecuencia)", "Q3")

# --- covariables de contexto -------------------------------------------------
registrar("sexo_c", df["sexo"].map(valores("sexo")), ["Hombre", "Mujer"], "sexo", "contexto")
registrar("edad_gpo", df["edad_1"].map(valores("edad_1")),
          [v for k, v in sorted(valores("edad_1").items())], "edad_1", "contexto")
registrar("tam_loc_c",
          df["tam_loc"].map({**valores("tam_loc"), 1: "100 000 y mas habitantes"}),
          ["100 000 y mas habitantes", "15 000-99 999 habitantes",
           "2 500-14 999 habitantes", "1-2499 habitantes"], "tam_loc", "contexto")
registrar("region_c",
          df["region"].map({**valores("region"), 1: "region_1_sin_documentar"}),
          ["region_1_sin_documentar", "Metropolitana", "Norte", "Sur"], "region", "contexto")

for nombre, (serie, niveles, origen, q) in NODOS.items():
    n_nulos = int(serie.isna().sum())
    print(f"{nombre:16} [{q:14}] nulos={n_nulos:4,}  " +
          "  ".join(f"{k}={v}" for k, v in serie.value_counts().items()))

escolaridad      [Q3            ] nulos=   1  basica=788  media_superior=314  superior=88
ingreso_gpo      [Q4            ] nulos=   1  sin_ingreso=589  mas_de_2_sm=421  hasta_2_sm=180
ingreso_bin      [Q4 (respaldo) ] nulos=   1  bajo=769  alto=421
perc_seguridad   [Q1            ] nulos=   0  inseguro=669  seguro=522
perc_eficiencia  [Q1            ] nulos=   0  eficiente=745  ineficiente=446
perc_costo       [Q2            ] nulos=   0  caro=616  barato=575
asalto_tp        [Q4            ] nulos=   3  no=1114  si=74
ocupacion        [Q2            ] nulos= 502  otro=552  patron_autoempleado=75  profesionista=62
grupo_tp         [Q1            ] nulos=   0  otro_tp=807  metro=210  no_tp=174
modo_principal   [Q3            ] nulos=   6  tp_concesionado=819  auto_moto=214  taxi=85  activo=25  otro=19  metro=10  foraneo=8  tp_masivo=5
sexo_c           [contexto      ] nulos=   0  Hombre=638  Mujer=553
edad_gpo         [contexto      ] nulos=   0  De 25 a 34 años=302  De 35 a 44 años=28

## 7. Ensamblado de la base final

In [10]:
IDENT = ["con1", "pondi2"]
AUDIT = ["match_informante", "usa_tp", "usa_metro", "modo_principal_empate"]

final = df[IDENT + list(NODOS) + AUDIT].copy()
final = final.rename(columns={"sexo_c": "sexo", "tam_loc_c": "tam_loc", "region_c": "region"})

print(f"Base final: {final.shape[0]:,} filas x {final.shape[1]} columnas\n")
print("Faltantes por nodo:")
nulos = final.isna().sum()
display(nulos[nulos > 0].to_frame("nulos").assign(
    pct=lambda t: (t["nulos"] / len(final) * 100).round(1)))
print("\nSin faltantes:", [c for c in final.columns if final[c].notna().all()])

Base final: 1,191 filas x 20 columnas

Faltantes por nodo:


,nulos,pct
escolaridad,1,0.1
ingreso_gpo,1,0.1
ingreso_bin,1,0.1
asalto_tp,3,0.3
ocupacion,502,42.1
modo_principal,6,0.5



Sin faltantes: ['con1', 'pondi2', 'perc_seguridad', 'perc_eficiencia', 'perc_costo', 'grupo_tp', 'sexo', 'edad_gpo', 'tam_loc', 'region', 'match_informante', 'usa_tp', 'usa_metro', 'modo_principal_empate']


## 8. Exportacion

Tres tipos de archivo:

1. **`enmt_bn_final.csv`** — la base completa, 1,191 filas, con `con1` (el ID real de encuestado;
   `folio` **no** lo es) y `pondi2` (el factor de expansion, por si se quieren estimaciones
   poblacionales y no solo muestrales).
2. **`enmt_bn_codebook.csv`** — que es cada nodo, de donde sale, sus niveles y su cobertura.
3. **`enmt_bn_q1..q4.csv`** — un archivo por query, ya filtrado a su universo y **sin `NA`**, porque
   `bnlearn` no admite faltantes al ajustar.

In [11]:
# R lee un CSV con `na.strings = "NA"` por defecto: si los faltantes se escriben como
# cadena vacia, `stringsAsFactors = TRUE` los convierte en un nivel de factor "" en vez de NA.
# Por eso se escribe "NA" explicito. Los booleanos van como TRUE/FALSE para que R los lea
# como logicos y no como factores de dos niveles.
BOOLEANAS = ["usa_tp", "usa_metro", "modo_principal_empate"]


def para_r(tabla):
    salida = tabla.copy()
    for c in BOOLEANAS:
        if c in salida.columns:
            salida[c] = salida[c].map({True: "TRUE", False: "FALSE"})
    return salida


RUTA_FINAL = DIR_PROC / "enmt_bn_final.csv"
para_r(final).to_csv(RUTA_FINAL, index=False, encoding="utf-8", na_rep="NA")

filas_cb = []
for nombre, (serie, niveles, origen, queries) in NODOS.items():
    nombre_exp = {"sexo_c": "sexo", "tam_loc_c": "tam_loc", "region_c": "region"}.get(nombre, nombre)
    conteos = serie.value_counts()
    # Solo los niveles que existen en los datos, conservando el orden declarado:
    # el codebook describe el archivo, no la escala teorica del cuestionario.
    niveles = [n for n in niveles if n in conteos.index]
    filas_cb.append({
        "nodo": nombre_exp,
        "origen": origen,
        "queries": queries,
        "n_niveles": len(niveles),
        "niveles_en_orden": " < ".join(niveles),
        "pct_nulos": round(float(serie.isna().mean()) * 100, 2),
        "n_por_nivel": "; ".join(f"{k}={conteos.get(k, 0)}" for k in niveles),
    })
codebook = pd.DataFrame(filas_cb)
RUTA_CB = DIR_PROC / "enmt_bn_codebook.csv"
codebook.to_csv(RUTA_CB, index=False, encoding="utf-8")

print(f"  escrito {RUTA_FINAL.name}  ({RUTA_FINAL.stat().st_size / 1024:.0f} KB)")
print(f"  escrito {RUTA_CB.name}")
display(codebook[["nodo", "queries", "n_niveles", "pct_nulos", "n_por_nivel"]])

  escrito enmt_bn_final.csv  (203 KB)
  escrito enmt_bn_codebook.csv


,nodo,queries,n_niveles,pct_nulos,n_por_nivel
0,escolaridad,Q3,3,0.08,basica=788; media_superior=314; superior=88
1,ingreso_gpo,Q4,3,0.08,sin_ingreso=589; hasta_2_sm=180; mas_de_2_sm=421
2,ingreso_bin,Q4 (respaldo),2,0.08,bajo=769; alto=421
3,perc_seguridad,Q1,2,0.00,seguro=522; inseguro=669
4,perc_eficiencia,Q1,2,0.00,eficiente=745; ineficiente=446
5,perc_costo,Q2,2,0.00,barato=575; caro=616
6,asalto_tp,Q4,2,0.25,no=1114; si=74
7,ocupacion,Q2,3,42.15,patron_autoempleado=75; profesionista=62; otro...
8,grupo_tp,Q1,3,0.00,metro=210; otro_tp=807; no_tp=174
9,modo_principal,Q3,8,0.50,tp_concesionado=819; auto_moto=214; taxi=85; a...


In [12]:
CONTEXTO = ["sexo", "edad_gpo", "tam_loc", "region"]   # sin faltantes, seguros para casos completos

QUERIES = {
    "q1": dict(
        titulo="Metro vs. resto de TP -> percepcion de seguridad y efectividad",
        universo=final["usa_tp"],
        nodos=["grupo_tp", "perc_seguridad", "perc_eficiencia"],
    ),
    "q2": dict(
        titulo="Patrones/autoempleados vs. profesionistas -> percepcion del costo",
        universo=final["ocupacion"].isin(["patron_autoempleado", "profesionista"]),
        nodos=["ocupacion", "perc_costo"],
    ),
    "q3": dict(
        titulo="Escolaridad -> modo principal",
        universo=pd.Series(True, index=final.index),
        nodos=["escolaridad", "modo_principal"],
    ),
    "q4": dict(
        titulo="Entre usuarios de TP: asalto en TP segun ingreso",
        universo=final["usa_tp"],
        nodos=["ingreso_gpo", "ingreso_bin", "asalto_tp"],
    ),
}

for clave, cfg in QUERIES.items():
    cols = ["con1", "pondi2"] + cfg["nodos"] + CONTEXTO
    sub = final.loc[cfg["universo"], cols]
    n_universo = len(sub)
    sub = sub.dropna()
    ruta = DIR_PROC / f"enmt_bn_{clave}.csv"
    para_r(sub).to_csv(ruta, index=False, encoding="utf-8", na_rep="NA")
    perdidas = n_universo - len(sub)
    print(f"  escrito {ruta.name}: n={len(sub):,}  (universo {n_universo:,}, "
          f"{perdidas:,} filas con algun NA descartadas)")
    print(f"     {cfg['titulo']}")

  escrito enmt_bn_q1.csv: n=1,017  (universo 1,017, 0 filas con algun NA descartadas)
     Metro vs. resto de TP -> percepcion de seguridad y efectividad


  escrito enmt_bn_q2.csv: n=137  (universo 137, 0 filas con algun NA descartadas)
     Patrones/autoempleados vs. profesionistas -> percepcion del costo
  escrito enmt_bn_q3.csv: n=1,184  (universo 1,191, 7 filas con algun NA descartadas)
     Escolaridad -> modo principal
  escrito enmt_bn_q4.csv: n=1,014  (universo 1,017, 3 filas con algun NA descartadas)
     Entre usuarios de TP: asalto en TP segun ingreso


## 9. Resultados: los cuatro queries

Las tablas cruzadas de cada query. **Los tamanios de celda son lo importante:** una celda con muy
pocos casos produce una probabilidad condicional que parece un resultado pero no lo es. Conviene
mirarlos antes de ajustar nada en R.

In [13]:
q1 = pd.read_csv(DIR_PROC / "enmt_bn_q1.csv")
print(f"=== Q1 — metro vs. resto de TP (n={len(q1):,}) ===")
display(pd.crosstab(q1["grupo_tp"], q1["perc_seguridad"], margins=True))
display(pd.crosstab(q1["grupo_tp"], q1["perc_eficiencia"], margins=True))

q2 = pd.read_csv(DIR_PROC / "enmt_bn_q2.csv")
print(f"\n=== Q2 — ocupacion y percepcion del costo (n={len(q2):,}) ===")
display(pd.crosstab(q2["ocupacion"], q2["perc_costo"], margins=True))

q3 = pd.read_csv(DIR_PROC / "enmt_bn_q3.csv")
print(f"\n=== Q3 — escolaridad y modo principal (n={len(q3):,}) ===")
display(pd.crosstab(q3["escolaridad"], q3["modo_principal"], margins=True))

q4 = pd.read_csv(DIR_PROC / "enmt_bn_q4.csv")
print(f"\n=== Q4 — asalto en TP segun ingreso (n={len(q4):,}) ===")
print("Version de 3 niveles (ingreso_gpo):")
display(pd.crosstab(q4["ingreso_gpo"], q4["asalto_tp"], margins=True))
print("Version binaria (ingreso_bin), respaldo si la de 3 niveles queda muy delgada:")
display(pd.crosstab(q4["ingreso_bin"], q4["asalto_tp"], margins=True))

=== Q1 — metro vs. resto de TP (n=1,017) ===


perc_seguridad,inseguro,seguro,All
grupo_tp,,,
metro,141,69,210
otro_tp,430,377,807
All,571,446,1017


perc_eficiencia,eficiente,ineficiente,All
grupo_tp,,,
metro,115,95,210
otro_tp,538,269,807
All,653,364,1017



=== Q2 — ocupacion y percepcion del costo (n=137) ===


perc_costo,barato,caro,All
ocupacion,,,
patron_autoempleado,34,41,75
profesionista,26,36,62
All,60,77,137


...

modo_principal,activo,auto_moto,foraneo,metro,otro,taxi,tp_concesionado,tp_masivo,All
escolaridad,,,,,,,,,
basica,21,104,4,8,15,69,557,5,783
media_superior,3,68,4,2,4,12,220,0,313
superior,1,42,0,0,0,4,41,0,88
All,25,214,8,10,19,85,818,5,1184



=== Q4 — asalto en TP segun ingreso (n=1,014) ===
Version de 3 niveles (ingreso_gpo):


asalto_tp,no,si,All
ingreso_gpo,,,
hasta_2_sm,155,9,164
mas_de_2_sm,317,33,350
sin_ingreso,472,28,500
All,944,70,1014


Version binaria (ingreso_bin), respaldo si la de 3 niveles queda muy delgada:


asalto_tp,no,si,All
ingreso_bin,,,
alto,317,33,350
bajo,627,37,664
All,944,70,1014


---

## Notas para el equipo

**Como leer la base en R.** Los nodos vienen como texto, asi que salen como factores directamente:

```r
d <- read.csv("data/processed/enmt_bn_final.csv", stringsAsFactors = TRUE)
q4 <- read.csv("data/processed/enmt_bn_q4.csv", stringsAsFactors = TRUE)
```

Los faltantes van escritos como `NA` y los indicadores logicos como `TRUE`/`FALSE`, que es
justo lo que `read.csv` espera: los nodos salen como factores sin niveles espurios y las columnas
de auditoria como `logical`.

El **orden de los niveles** no se conserva en un CSV: `enmt_bn_codebook.csv` trae la columna
`niveles_en_orden` con el orden previsto de cada nodo. Para los ordinales (`escolaridad`,
`ingreso_gpo`, `edad_gpo`) conviene fijarlo con `factor(..., levels = ..., ordered = TRUE)` antes de
modelar. Los archivos `q1..q4` ya vienen filtrados y **sin `NA`**, que es lo que `bnlearn` necesita.

**Decisiones que este notebook tomo y se pueden discutir.** Ninguna esta escondida: todas son un
diccionario o una lista al principio de su celda, y cambiarlas es re-ejecutar.

- **Patrones agrupados con autoempleados.** Los patrones solos son demasiado pocos para sostener una
  distribucion condicional. Se agruparon con los trabajadores por cuenta propia, que comparten lo
  relevante para el query: no reciben salario de un tercero. Es una decision sustantiva, no tecnica.
- **El ingreso se trata como ordinal agrupado, no como monto.** `ing_ind` son rangos de salarios
  minimos; sacarles un promedio no significa nada. Por eso el query cambia de "≷ el promedio" a
  "por nivel de ingreso". Se exportan **dos versiones** (`ingreso_gpo` de 3 niveles e `ingreso_bin`
  de 2) porque con solo 74 asaltos en toda la muestra el corte de 3 niveles puede dejar celdas
  demasiado delgadas — la tabla cruzada de la seccion 10 muestra si aguanta.
- **El universo de TP se define por uso, no por modo principal.** Quien usa el metro a diario pero
  maneja mas tiene una opinion del TP que el query necesita.
- **El autobus foraneo no cuenta como TP urbano**, porque `p17_*` pregunta por el transporte
  publico "en su localidad o ciudad".
- **El desempate del modo principal se deriva de los datos** (gana el grupo con mas usuarios
  cotidianos), no de un orden elegido a mano. Pero **el empate es la norma, no la excepcion**: 454 de
  1,185 personas (38 %) declaran la misma frecuencia maxima en dos o mas modos. Como la regla da la
  victoria al grupo mas usado, `tp_concesionado` absorbe casi todos los empates y termina con el
  69 % de los casos, mientras que `metro` aparece como modo principal solo 10 veces. Dos consecuencias
  practicas:
    - **Para cualquier pregunta sobre el metro, usar `grupo_tp`, no `modo_principal`.** Por eso Q1 se
      construye con `grupo_tp`, que tiene 210 usuarios de metro en lugar de 10.
    - Las filas empatadas estan marcadas en `modo_principal_empate`. Excluirlas es la prueba de
      robustez natural: al hacerlo en Q3 (n baja de 1,184 a 730) el patron no solo se mantiene sino
      que se acentua — el uso del automovil como modo principal pasa de 13/22/48 % a 18/32/65 %
      segun escolaridad basica/media superior/superior.
- **Los codigos sin etiqueta se resolvieron con evidencia, no por intuicion** (seccion 3). El unico
  que quedo sin resolver, `region = 1`, se dejo nombrado como tal.

**Lo que este notebook NO hizo, a proposito.**

- **No imputo nada.** Se mantiene el criterio de la etapa de limpieza: si un analisis necesita
  imputacion, que la haga y la documente, para no arrastrar supuestos ajenos.
- **No elimino filas de la base principal.** `enmt_bn_final.csv` conserva los 1,191 encuestados con
  sus `NA`. El filtrado a casos completos ocurre solo en los archivos por query, que es donde hace
  falta.
- **No aplico los factores de expansion.** `pondi2` viaja como columna. Las tablas de la seccion 10
  son **muestrales**; para hablar de la poblacion hay que ponderar.

**El punto fragil es Q4.** Solo 74 personas en toda la muestra reportaron un asalto en transporte
publico, y el universo de TP es mas chico todavia. Es la restriccion de tamanio mas seria del
proyecto: conviene reportar los intervalos de las probabilidades condicionales, no solo los puntos.